```
# Lab type:  prompt
# Course:    NL301 Natural Language Processing with Python
# Lesson:    08 — Semantic Similarity and Vector Search
# Task:      Complete a FAISS retrieval pipeline with bi-encoder + cross-encoder re-ranking.
```

## Setup

In [ ]:
!pip install faiss-cpu sentence-transformers --quiet
import numpy as np
import faiss
from sentence_transformers import SentenceTransformer, CrossEncoder

bi_encoder = SentenceTransformer('all-MiniLM-L6-v2')

# Document corpus
corpus = [
    "Python list comprehensions provide a concise way to create lists.",
    "FastAPI is a modern high-performance web framework for building APIs.",
    "Pandas DataFrames allow tabular data manipulation with labelled axes.",
    "FAISS enables efficient similarity search over millions of dense vectors.",
    "Sentence transformers produce fixed-size embeddings from variable-length text.",
    "Cross-encoders jointly encode query and document for more accurate relevance scoring.",
    "NumPy provides multi-dimensional array operations optimised for numerical computing.",
    "Logistic regression is a linear classifier that outputs class probabilities.",
    "Cosine similarity measures the angle between two vectors regardless of magnitude.",
    "Transfer learning fine-tunes a pretrained model on a smaller domain-specific dataset.",
]
queries = [
    "how to search vectors efficiently",
    "reranking results with a cross encoder",
    "Python data manipulation library",
]


---
## Task 1: Build the encode + index function

Complete `build_index` to encode the corpus, apply L2 normalisation (critical for `IndexFlatIP` to compute cosine similarity), and add to a FAISS index.

In [ ]:
def build_index(corpus: list[str]) -> tuple[faiss.IndexFlatIP, np.ndarray]:
    """Encode corpus, normalise, and build a FAISS flat inner-product index."""
    # TODO:
    # 1. Encode corpus with bi_encoder (return numpy arrays)
    # 2. Cast to float32
    # 3. Normalise with faiss.normalize_L2 (IMPORTANT — without this,
    #    IndexFlatIP computes dot product, not cosine similarity)
    # 4. Create faiss.IndexFlatIP with the embedding dimension
    # 5. Add embeddings to the index
    # 6. Return (index, embeddings)
    
    embeddings = None   # replace with your code
    index = None        # replace with your code
    return index, embeddings

index, corpus_embs = build_index(corpus)
print(f"Index built: {index.ntotal} vectors, dim={index.d}")


**Critical note:** What happens if you skip `faiss.normalize_L2` before using `IndexFlatIP`? Run a quick experiment — compare cosine similarities with and without normalisation for a single query.

*(Write your answer here.)*

---
## Task 2: Write the query function

Complete `search` to encode the query, normalise, call `index.search`, and return ranked `(score, document)` pairs.

In [ ]:
def search(query: str, index: faiss.IndexFlatIP, corpus: list[str],
           top_k: int = 5) -> list[tuple[float, str]]:
    """Return top_k (score, document) pairs for a query."""
    # TODO:
    # 1. Encode query with bi_encoder
    # 2. Cast to float32 and reshape to (1, dim)
    # 3. Normalise with faiss.normalize_L2
    # 4. Call index.search(query_emb, top_k) → returns (distances, indices)
    # 5. Return [(score, corpus[i]) for i, score in zip(indices[0], distances[0])]
    
    pass  # replace with your code

for q in queries:
    print(f"\nQuery: '{q}'")
    results = search(q, index, corpus, top_k=3)
    if results:
        for score, doc in results:
            print(f"  {score:.4f}  {doc[:70]}")


---
## Task 3: Add cross-encoder re-ranking

Bi-encoders are fast but approximate. A cross-encoder reads query + document together for more accurate relevance. Add re-ranking on top of the top-10 bi-encoder results.

In [ ]:
cross_encoder = CrossEncoder('cross-encoder/ms-marco-MiniLM-L-6-v2')

def search_with_rerank(query: str, index: faiss.IndexFlatIP, corpus: list[str],
                       bi_top_k: int = 10, rerank_top_k: int = 3) -> list[tuple[float, str]]:
    """Retrieve bi_top_k candidates, re-rank with cross-encoder, return rerank_top_k."""
    # TODO:
    # 1. Call search() to get top bi_top_k candidates
    # 2. Build pairs: [[query, doc] for _, doc in candidates]
    # 3. Score with cross_encoder.predict(pairs)
    # 4. Sort by cross-encoder score descending
    # 5. Return top rerank_top_k (cross_score, doc) pairs
    
    pass  # replace with your code

test_query = "how to search vectors efficiently"
print("=== Bi-encoder only (top 3) ===")
for score, doc in (search(test_query, index, corpus, top_k=3) or []):
    print(f"  {score:.4f}  {doc[:70]}")

print("\n=== After cross-encoder re-ranking ===")
for score, doc in (search_with_rerank(test_query, index, corpus) or []):
    print(f"  {score:.4f}  {doc[:70]}")


**Analysis questions:**

1. At what corpus size (number of documents) would you switch from `IndexFlatIP` (exact search) to `IndexIVFFlat` (approximate search)? What trade-off does `IndexIVFFlat` make?
2. What is the difference between *relevance* (what the user wants) and *similarity* (vector proximity)? Give a concrete example where high similarity ≠ high relevance.

*(Write your answers here.)*